# Self-Hosted NeuroCore Hub Evidence

This notebook exercises SC-NeuroCore's offline-first self-hosted hub bundle generator: local model-zoo index, manifest contracts, Docker Compose hardening, opt-in benchmark plan, and bundle file generation.

## Evidence Boundary

This notebook generates and validates local bundle artefacts only. It does not start Docker, build a container image, expose a production service, prove network isolation, or certify operational security. Production deployment still requires container build provenance, runtime scans, host hardening, secret management, TLS, monitoring, and acceptance tests in the target environment.

In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import yaml

from sc_neurocore.hub import (
    HubBundleConfig,
    build_benchmark_plan,
    build_hub_manifest,
    build_model_zoo_index,
    write_hub_bundle,
)

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

model_zoo_index = build_model_zoo_index()
model_zoo_summary = {
    "schema_version": model_zoo_index["schema_version"],
    "plugins": [plugin["name"] for plugin in model_zoo_index["plugins"]],
    "network_count": len(model_zoo_index["network_configs"]),
    "pretrained": [entry["name"] for entry in model_zoo_index["pretrained"]],
}
assert model_zoo_summary["plugins"] == ["AdEx", "Hodgkin-Huxley", "Izhikevich", "LIF"]
assert {"mnist", "shd", "dvs_gesture"}.issubset(model_zoo_summary["pretrained"])
model_zoo_summary

In [ ]:
config = HubBundleConfig(studio_port=8123, offline=True)
manifest = build_hub_manifest(config)
manifest_summary = {
    "schema_version": manifest["schema_version"],
    "studio_url": manifest["services"]["studio"]["url"],
    "readiness_endpoint": manifest["service_contracts"]["studio"]["readiness_endpoint"],
    "ingress_scope": manifest["network_policy"]["ingress_scope"],
    "external_egress_required": manifest["network_policy"]["external_egress_required"],
    "offline_environment": manifest["network_policy"]["offline_environment"],
    "hardening": manifest["container_hardening"],
    "studio_does_not_provide": manifest["service_contracts"]["studio"]["does_not_provide"],
}
assert manifest_summary["schema_version"] == "sc-neurocore.self-hosted-hub.v1"
assert manifest_summary["ingress_scope"] == "loopback"
assert manifest_summary["external_egress_required"] is False
assert manifest_summary["offline_environment"]["SC_NEUROCORE_HUB_OFFLINE"] == "1"
assert manifest_summary["hardening"]["read_only_root_filesystem"] is True
assert "hardware or cloud job submission" in manifest_summary["studio_does_not_provide"]
manifest_summary

In [ ]:
private_manifest = build_hub_manifest(HubBundleConfig(bind_host="10.10.0.5", offline=False))
all_interfaces_manifest = build_hub_manifest(HubBundleConfig(bind_host="0.0.0.0"))
ingress_summary = {
    "private_network_scope": private_manifest["network_policy"]["ingress_scope"],
    "private_offline_flag": private_manifest["network_policy"]["offline_environment"]["SC_NEUROCORE_HUB_OFFLINE"],
    "all_interfaces_scope": all_interfaces_manifest["network_policy"]["ingress_scope"],
}
assert ingress_summary["private_network_scope"] == "private_network"
assert ingress_summary["private_offline_flag"] == "0"
assert ingress_summary["all_interfaces_scope"] == "all_interfaces"
ingress_summary

In [ ]:
benchmark_plan = build_benchmark_plan(HubBundleConfig(benchmarks_dir="bench"))
benchmark_summary = {
    "schema_version": benchmark_plan["schema_version"],
    "profile": benchmark_plan["runner"]["profile"],
    "results_dir": benchmark_plan["runner"]["results_dir"],
    "mounted_benchmarks": benchmark_plan["mounted_paths"]["benchmarks"],
    "limitations": benchmark_plan["limitations"],
}
assert benchmark_summary["profile"] == "benchmark"
assert "not started with the Studio service" in benchmark_summary["limitations"][1]
benchmark_summary

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    output_dir = Path(tmp)
    paths = write_hub_bundle(
        output_dir,
        HubBundleConfig(studio_port=9000, cache_dir="local-cache", benchmarks_dir="bench"),
    )
    compose_text = paths["compose"].read_text(encoding="utf-8")
    compose_doc = yaml.safe_load(compose_text)
    written_manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
    written_index = json.loads(paths["model_zoo_index"].read_text(encoding="utf-8"))
    bundle_summary = {
        "generated_keys": sorted(paths),
        "cache_dir_created": (output_dir / "local-cache").is_dir(),
        "models_dir_created": (output_dir / "models").is_dir(),
        "benchmark_results_dir_created": (output_dir / "bench" / "results").is_dir(),
        "studio_port_mapping": compose_doc["services"]["studio"]["ports"][0],
        "studio_read_only": compose_doc["services"]["studio"]["read_only"],
        "benchmark_profiles": compose_doc["services"]["benchmark-runner"]["profiles"],
        "network_driver": compose_doc["networks"]["neurocore-hub"]["driver"],
        "manifest_index_matches": written_index == written_manifest["model_zoo"],
        "readme_ends_with_newline": paths["readme"].read_text(encoding="utf-8").endswith("\n"),
    }

assert bundle_summary["generated_keys"] == [
    "benchmark_plan",
    "compose",
    "env_example",
    "manifest",
    "model_zoo_index",
    "readme",
]
assert bundle_summary["studio_port_mapping"] == "127.0.0.1:9000:9000"
assert bundle_summary["studio_read_only"] is True
assert bundle_summary["benchmark_profiles"] == ["benchmark"]
assert bundle_summary["manifest_index_matches"] is True
bundle_summary

In [ ]:
guardrail_inputs = [
    ({"bind_host": ""}, "bind_host must not be empty"),
    ({"studio_port": 0}, "studio_port must be in the range"),
    ({"cache_dir": "/tmp/cache"}, "cache_dir must be"),
    ({"models_dir": "../models"}, "models_dir must be"),
    ({"bind_host": "127.0.0.1:8001"}, "bind_host must be"),
    ({"compose_name": "nested/docker-compose.yml"}, "compose_name must be a file name"),
]
guardrail_summary = {}
for kwargs, expected in guardrail_inputs:
    try:
        HubBundleConfig(**kwargs)
    except ValueError as exc:
        message = str(exc)
    else:
        raise AssertionError(f"invalid HubBundleConfig accepted: {kwargs}")
    assert expected in message
    guardrail_summary[str(kwargs)] = message
guardrail_summary

In [ ]:
manifest_evidence = {
    "schema_version": "sc-neurocore.self-hosted-hub-evidence.v1",
    "model_zoo_summary": model_zoo_summary,
    "manifest_summary": manifest_summary,
    "ingress_summary": ingress_summary,
    "benchmark_summary": benchmark_summary,
    "bundle_summary": bundle_summary,
    "guardrails": guardrail_summary,
    "evidence_boundary": "Local bundle generation only; no Docker start, image build, production exposure, network-isolation proof, or operational-security certification claim.",
}
manifest_evidence